In [1]:
import pandas as pd

# Load dataset
train_df = pd.read_csv("/content/phm_train.csv")
test_df = pd.read_csv("/content/phm_test.csv")

print(train_df.head())
print(train_df['label'].value_counts())


       tweet_id  label                                              tweet
0  6.430000e+17      0  user_mention all i can tell you is i have had ...
1  6.440000e+17      0  my doctor told me stop he gave me sum pop i mi...
2  8.150000e+17      1  i take tylenol and i wake up in the middle of ...
3  6.820000e+17      0  i got xans in an advil bottle i dont take them...
4  6.440000e+17      1  mom says i need to stop eating so much bc ive ...
label
0    7091
1    2900
Name: count, dtype: int64


In [2]:
print(train_df['label'].value_counts(normalize=True))


label
0    0.709739
1    0.290261
Name: proportion, dtype: float64


In [3]:
import re

def clean_tweet(text):
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove mentions
    text = re.sub(r'@\w+', '', text)

    # Remove hashtags symbol only
    text = re.sub(r'#', '', text)

    # Remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

train_df['clean_text'] = train_df['tweet'].apply(clean_tweet)
test_df['clean_text'] = test_df['tweet'].apply(clean_tweet)


In [4]:
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')

nltk.download('punkt_tab')


train_df['tokens'] = train_df['clean_text'].apply(word_tokenize)
test_df['tokens'] = test_df['clean_text'].apply(word_tokenize)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [5]:
from collections import Counter

# Build vocabulary
counter = Counter()

for tokens in train_df['tokens']:
    counter.update(tokens)

# Minimum frequency threshold
min_freq = 2

vocab = {word for word, freq in counter.items() if freq >= min_freq}

# Add special tokens
word2idx = {'<PAD>': 0, '<UNK>': 1}

for word in vocab:
    word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}

vocab_size = len(word2idx)
print("Vocabulary size:", vocab_size)


Vocabulary size: 5442


In [6]:
def encode(tokens, word2idx):
    return [word2idx.get(word, word2idx['<UNK>']) for word in tokens]

train_df['encoded'] = train_df['tokens'].apply(lambda x: encode(x, word2idx))
test_df['encoded'] = test_df['tokens'].apply(lambda x: encode(x, word2idx))


In [7]:
from torch.nn.utils.rnn import pad_sequence
import torch

MAX_LEN = 50

def pad_sequence_custom(seq, max_len):
    if len(seq) > max_len:
        return seq[:max_len]
    else:
        return seq + [0] * (max_len - len(seq))

train_padded = [pad_sequence_custom(seq, MAX_LEN) for seq in train_df['encoded']]
test_padded = [pad_sequence_custom(seq, MAX_LEN) for seq in test_df['encoded']]

X_train = torch.tensor(train_padded)
X_test = torch.tensor(test_padded)

y_train = torch.tensor(train_df['label'].values, dtype=torch.float32)
y_test = torch.tensor(test_df['label'].values, dtype=torch.float32)


In [8]:
!wget http://nlp.stanford.edu/data/glove.twitter.27B.zip
!unzip glove.twitter.27B.zip


--2026-02-19 11:35:19--  http://nlp.stanford.edu/data/glove.twitter.27B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.twitter.27B.zip [following]
--2026-02-19 11:35:20--  https://nlp.stanford.edu/data/glove.twitter.27B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.twitter.27B.zip [following]
--2026-02-19 11:35:20--  https://downloads.cs.stanford.edu/nlp/data/glove.twitter.27B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1520408563 (1.4G) [ap

In [9]:
import numpy as np

embedding_dim = 200

# Load GloVe
glove_path = "glove.twitter.27B.200d.txt"

embeddings_index = {}

with open(glove_path, 'r', encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = vector

print("Loaded GloVe vectors:", len(embeddings_index))


Loaded GloVe vectors: 1193514


In [10]:
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, idx in word2idx.items():
    vector = embeddings_index.get(word)
    if vector is not None:
        embedding_matrix[idx] = vector
    else:
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

embedding_matrix = torch.tensor(embedding_matrix, dtype=torch.float32)


In [11]:
from torch.utils.data import Dataset, DataLoader

class TweetDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 32

train_dataset = TweetDataset(X_train, y_train)
test_dataset = TweetDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


In [12]:
print(embedding_matrix.shape)
print(vocab_size)
print(embedding_dim)
print(len(train_loader))


torch.Size([5442, 200])
5442
200
313


Layer1


In [13]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader

# Split training into train + validation
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    stratify=y_train,
    random_state=42
)

# Convert to PyTorch tensors
train_dataset = TensorDataset(
    torch.tensor(X_train_final, dtype=torch.long),
    torch.tensor(y_train_final, dtype=torch.float32)
)

val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.long),
    torch.tensor(y_val, dtype=torch.float32)
)

test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.long),
    torch.tensor(y_test, dtype=torch.float32)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)


/tmp/ipython-input-2634185891.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(X_train_final, dtype=torch.long),
/tmp/ipython-input-2634185891.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(y_train_final, dtype=torch.float32)
/tmp/ipython-input-2634185891.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(X_val, dtype=torch.long),
/tmp/ipython-input-2634185891.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [15]:
import torch.nn as nn
import torch.nn.functional as F

class LSTM_Attention(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, embedding_matrix):
        super(LSTM_Attention, self).__init__()

        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix,
            freeze=True
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.attention = nn.Linear(hidden_dim, 1)

        self.dropout = nn.Dropout(0.4)   # Reduced from 0.5
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)

        attn_weights = torch.tanh(self.attention(lstm_out))
        attn_weights = F.softmax(attn_weights, dim=1)

        context = torch.sum(attn_weights * lstm_out, dim=1)

        out = self.dropout(context)
        out = self.fc(out)

        return out


In [16]:
hidden_dim = 256

model = LSTM_Attention(vocab_size, embedding_dim, hidden_dim, embedding_matrix)
model = model.to(device)


In [17]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Ensure numpy array
y_train_np = np.array(y_train_final)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),   # Explicitly define classes
    y=y_train_np
)

print("Class weights:", class_weights)

pos_weight = torch.tensor([class_weights[1]], dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


Class weights: [0.70449643 1.72251521]


In [18]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)


In [19]:
from sklearn.metrics import accuracy_score, classification_report

def evaluate(model, loader, threshold=0.5):

    model.eval()

    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for X_batch, y_batch in loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            probs = torch.sigmoid(outputs)

            all_probs.extend(probs.cpu().numpy())

            preds = (probs > threshold).float()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    all_preds = np.array(all_preds).flatten()
    all_labels = np.array(all_labels).flatten()
    all_probs = np.array(all_probs).flatten()

    acc = accuracy_score(all_labels, all_preds)

    print("Test Accuracy:", round(acc * 100, 2), "%")
    print("\nClassification Report:\n")
    print(classification_report(all_labels, all_preds, digits=4))

    return acc


In [20]:
def train_model(model, train_loader, val_loader, epochs=20):

    best_acc = 0
    patience = 3
    counter = 0

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device).unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"\nEpoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

        val_acc = evaluate(model, val_loader)

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_lstm_attention.pt")
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            print("\nEarly stopping triggered.")
            break

    print("\nBest Validation Accuracy:", round(best_acc * 100, 2), "%")


In [21]:
def train_model(model, train_loader, val_loader, epochs=20):

    best_acc = 0
    patience = 3
    counter = 0

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device).unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"\nEpoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

        val_acc = evaluate(model, val_loader)

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_lstm_attention.pt")
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            print("\nEarly stopping triggered.")
            break

    print("\nBest Validation Accuracy:", round(best_acc * 100, 2), "%")


In [22]:
train_model(model, train_loader, val_loader, epochs=20)



Epoch 1, Loss: 0.69735102467519
Test Accuracy: 82.72 %

Classification Report:

              precision    recall  f1-score   support

         0.0     0.8414    0.9323    0.8845      1064
         1.0     0.7750    0.5701    0.6570       435

    accuracy                         0.8272      1499
   macro avg     0.8082    0.7512    0.7707      1499
weighted avg     0.8221    0.8272    0.8185      1499


Epoch 2, Loss: 0.5615669510194233
Test Accuracy: 82.32 %

Classification Report:

              precision    recall  f1-score   support

         0.0     0.9089    0.8346    0.8702      1064
         1.0     0.6628    0.7954    0.7231       435

    accuracy                         0.8232      1499
   macro avg     0.7859    0.8150    0.7966      1499
weighted avg     0.8375    0.8232    0.8275      1499


Epoch 3, Loss: 0.5129488749163491
Test Accuracy: 81.72 %

Classification Report:

              precision    recall  f1-score   support

         0.0     0.9140    0.8195    0.8642 